# E001 — Audio baseline: MFCC + GMM

See `experiments/E001_audio-mfcc-gmm-baseline.md` for hypothesis and setup.

**Pipeline:**
1. Load manifest
2. For each LOSO fold: extract MFCC, apply CMN, train two GMMs, score val fold
3. Evaluate OOF scores (EER, min-DCF per fold + mean ± std)
4. Rich visualisations: MFCC spectrograms, score distributions, ROC, DET, per-fold bar chart, calibration panel

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve, auc
from scipy import stats

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf, evaluate

# ── consistent style ────────────────────────────────────────────────────
C_TARGET    = "#E74C3C"   # tomato
C_NONTARGET = "#2E86AB"   # steelblue
C_GREEN     = "#27AE60"   # green  (threshold / good)
C_PURPLE    = "#8E44AD"   # purple (EER operating point)

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "font.size": 10,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. MFCC Feature Extraction

**MFCC** (Mel-frequency cepstral coefficients) are the standard features for speaker
verification. They capture the shape of the vocal tract — *who* is speaking, not *what*
is being said.

**CMN** (Cepstral Mean Normalization): subtract the per-utterance mean of each
coefficient. This removes the effect of the microphone and room — otherwise the model
would recognise the room rather than the person.

Output: matrix `(T, 13)` where T ≈ number of frames (~100 frames/second).

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    """Locate WAV by searching all four corpus sub-folders."""
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(f"WAV not found for stem: {stem}")


def extract_mfcc(wav_path: Path, n_mfcc: int = 13) -> np.ndarray:
    """
    Return MFCC matrix shape (T, n_mfcc) with CMN applied.
    Each row = one frame (~10 ms).
    """
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)  # (n_mfcc, T)
    mfcc = mfcc.T                                             # (T, n_mfcc)
    mfcc -= mfcc.mean(axis=0)                                 # CMN
    return mfcc


def extract_mfcc_batch(df: pd.DataFrame, data_dir: Path, n_mfcc: int = 13):
    """
    Extract MFCC for all utterances in df.
    Returns concatenated (N_frames, n_mfcc) matrix and frame-level labels.
    """
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        mfcc = extract_mfcc(find_wav(row["stem"], data_dir), n_mfcc)
        all_mfcc.append(mfcc)
        all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


# Quick sanity check
sample_target = manifest[manifest.label == 1].iloc[0]
sample_nontarget = manifest[manifest.label == 0].iloc[0]
mfcc_target    = extract_mfcc(find_wav(sample_target["stem"], DATA))
mfcc_nontarget = extract_mfcc(find_wav(sample_nontarget["stem"], DATA))

print(f"Target     sample: {sample_target['stem']}  → MFCC shape {mfcc_target.shape}")
print(f"Non-target sample: {sample_nontarget['stem']} → MFCC shape {mfcc_nontarget.shape}")

### 1.1 MFCC Spectrograms: target vs. non-target

Each column is a 10 ms frame; each row is one of the 13 cepstral coefficients.
After CMN the colour scale is centred around zero — blue = below mean, yellow/white = above.
Even to the naked eye the two speakers show distinct texture patterns, which the GMM will
learn to separate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

for ax, mfcc, title, color in zip(
    axes,
    [mfcc_target, mfcc_nontarget],
    [f"Target ({sample_target['stem']})", f"Non-target ({sample_nontarget['stem']})"],
    [C_TARGET, C_NONTARGET],
):
    vmax = np.abs(mfcc).max()
    img = ax.imshow(
        mfcc.T, aspect="auto", origin="lower",
        cmap="magma", vmin=-vmax, vmax=vmax,
        interpolation="nearest",
    )
    ax.set_xlabel("Frame (≈10 ms each)")
    ax.set_ylabel("MFCC coefficient index")
    ax.set_title(title, color=color, fontweight="bold")
    ax.set_yticks(range(13))
    plt.colorbar(img, ax=ax, shrink=0.9, label="Amplitude (after CMN)")

fig.suptitle("MFCC Spectrograms — target vs. non-target (13 coefficients, CMN applied)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. GMM Training

**GMM** (Gaussian Mixture Model) models the probability density in MFCC space.
We train **two** — one for the target speaker, one for non-target.

The score for one utterance is the mean log-likelihood ratio (LLR) across its frames:

$$\text{score}(x) = \frac{1}{T}\sum_{t=1}^{T}\bigl[\log P(\text{mfcc}_t\mid\text{GMM}_{\text{target}}) - \log P(\text{mfcc}_t\mid\text{GMM}_{\text{nontarget}})\bigr]$$

Why mean and not sum? So that utterance length does not inflate the score.

Component counts: **8 for target** (few training samples — avoid overfitting),
**32 for non-target** (much more data available).

In [ ]:
def train_gmm(X: np.ndarray, n_components: int, seed: int = 67) -> GaussianMixture:
    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="diag",  # diagonal covariance: fewer params, less overfitting
        max_iter=200,
        random_state=seed,
    )
    gmm.fit(X)
    return gmm


def score_utterance(
    wav_path: Path,
    gmm_target: GaussianMixture,
    gmm_nontarget: GaussianMixture,
) -> float:
    """Mean LLR score for one utterance. Higher = more likely target."""
    mfcc = extract_mfcc(wav_path)
    llr = gmm_target.score_samples(mfcc) - gmm_nontarget.score_samples(mfcc)
    return float(llr.mean())

## 3. LOSO Cross-validation — OOF Scores

**Leave-One-Session-Out (LOSO):** target has 3 recording sessions → 3 folds.
Non-target speakers are distributed round-robin across folds (identity-disjoint).

For each fold:
1. Extract MFCC from **train fold only** (no leakage)
2. Train `GMM_target` (8 components) and `GMM_nontarget` (32 components)
3. Score every utterance in the val fold → store as OOF score

CMN is computed per-utterance inside `extract_mfcc`, so there is no
cross-contamination between train and val statistics.

In [ ]:
N_MFCC                = 13
N_TARGET_COMPONENTS   = 8
N_NONTARGET_COMPONENTS = 32
SEED                  = 67

oof_scores  = np.full(len(manifest), np.nan)
fold_results = []
fold_val_data = {}   # fold_id -> (scores_array, labels_array) — stored for per-fold viz

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]

    print(f"Fold {fold_id}: train={len(train_df)}, val={len(val_df)} "
          f"(target val: {(val_df.label==1).sum()}, non-target val: {(val_df.label==0).sum()})")

    # Feature extraction — train fold only
    print(f"  Extracting MFCC for train...")
    X_train, y_train = extract_mfcc_batch(train_df, DATA)

    X_target    = X_train[y_train == 1]
    X_nontarget = X_train[y_train == 0]
    print(f"  Train frames: {len(X_target)} target, {len(X_nontarget)} non-target")

    # Train GMMs
    print(f"  Training GMMs...")
    gmm_t  = train_gmm(X_target,    n_components=N_TARGET_COMPONENTS,    seed=SEED)
    gmm_nt = train_gmm(X_nontarget, n_components=N_NONTARGET_COMPONENTS, seed=SEED)

    # Score val fold
    print(f"  Scoring val fold...")
    for idx, row in val_df.iterrows():
        oof_scores[idx] = score_utterance(find_wav(row["stem"], DATA), gmm_t, gmm_nt)

    # Per-fold metrics
    val_scores = oof_scores[val_idx]
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    fold_val_data[fold_id] = (val_scores.copy(), val_labels.copy())

    eer, _     = compute_eer(val_scores[val_labels==1], val_scores[val_labels==0])
    min_dcf, _ = compute_min_dcf(val_scores[val_labels==1], val_scores[val_labels==0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
    print(f"  → EER = {eer*100:.2f}%, min-DCF = {min_dcf:.4f}\n")

print("Done.")

## 4. Results Table

In [ ]:
results_df = pd.DataFrame(fold_results)
mean_eer   = results_df["eer"].mean()
std_eer    = results_df["eer"].std()
mean_dcf   = results_df["min_dcf"].mean()
std_dcf    = results_df["min_dcf"].std()

print("=" * 45)
print(f"{'Fold':<8} {'EER [%]':>10} {'min-DCF':>10}")
print("-" * 45)
for _, r in results_df.iterrows():
    print(f"{int(r.fold):<8} {r.eer*100:>10.2f} {r.min_dcf:>10.4f}")
print("-" * 45)
print(f"{'mean±std':<8} {mean_eer*100:>7.2f}±{std_eer*100:.2f} {mean_dcf:>7.4f}±{std_dcf:.4f}")
print("=" * 45)

# OOF overall
y_all = manifest["label"].to_numpy()
eer_oof, _      = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])
dcf_oof, thr_oof = compute_min_dcf(oof_scores[y_all==1], oof_scores[y_all==0])
eer_oof_thr     = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])[1]

print(f"\nOOF overall: EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}, threshold = {thr_oof:.3f}")

## 5. Score Distributions per Fold

Each panel shows the histogram of LLR scores for target (tomato) and non-target
(steelblue) utterances in that fold's validation set. The green dashed line marks the
min-DCF threshold derived from OOF scores overall.

Ideal system: two well-separated distributions with minimal overlap.
The overlap region is where errors occur — targets scored below threshold (misses)
and non-targets above (false alarms).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes_flat = axes.flatten()

# Shared x-range across all panels
score_min = np.nanmin(oof_scores)
score_max = np.nanmax(oof_scores)
bins = np.linspace(score_min, score_max, 35)

# Per-fold panels (0, 1, 2)
for fold_id, ax in enumerate(axes_flat[:3]):
    val_scores, val_labels = fold_val_data[fold_id]
    s_t  = val_scores[val_labels == 1]
    s_nt = val_scores[val_labels == 0]

    fold_eer, fold_thr = compute_eer(s_t, s_nt)

    ax.hist(s_nt, bins=bins, alpha=0.65, color=C_NONTARGET, label=f"Non-target (n={len(s_nt)})", density=True)
    ax.hist(s_t,  bins=bins, alpha=0.75, color=C_TARGET,    label=f"Target (n={len(s_t)})",     density=True)
    ax.axvline(thr_oof, color=C_GREEN,  ls="--", lw=2, label=f"OOF threshold ({thr_oof:.2f})")
    ax.axvline(fold_thr, color=C_PURPLE, ls=":", lw=1.5, label=f"Fold EER thr ({fold_thr:.2f})")

    ax.set_title(f"Fold {fold_id}  |  EER = {fold_eer*100:.1f}%", fontweight="bold")
    ax.set_xlabel("LLR score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

# Overall OOF panel
ax = axes_flat[3]
s_t_all  = oof_scores[y_all == 1]
s_nt_all = oof_scores[y_all == 0]
ax.hist(s_nt_all, bins=bins, alpha=0.65, color=C_NONTARGET, label=f"Non-target (n={len(s_nt_all)})", density=True)
ax.hist(s_t_all,  bins=bins, alpha=0.75, color=C_TARGET,    label=f"Target (n={len(s_t_all)})",     density=True)
ax.axvline(thr_oof, color=C_GREEN, ls="--", lw=2, label=f"min-DCF threshold ({thr_oof:.2f})")
ax.set_title(f"Overall OOF  |  EER = {eer_oof*100:.1f}%, min-DCF = {dcf_oof:.4f}", fontweight="bold")
ax.set_xlabel("LLR score")
ax.set_ylabel("Density")
ax.legend(fontsize=8)

fig.suptitle("Score Distributions: Target vs. Non-target — per fold + overall OOF",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. ROC Curve

The **Receiver Operating Characteristic** plots True Positive Rate (1−FRR) against
False Positive Rate (FAR) as the decision threshold sweeps across all score values.

- **AUC** (Area Under Curve): 1.0 = perfect, 0.5 = random. Higher is better.
- The **EER operating point** (purple dot) is where FAR = FRR. It lies on the
  anti-diagonal and is the standard evaluation point for speaker verification.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_all, oof_scores)
roc_auc = auc(fpr, tpr)

# EER operating point on ROC
far_roc = fpr
frr_roc = 1.0 - tpr
eer_idx = np.argmin(np.abs(far_roc - frr_roc))
eer_far = far_roc[eer_idx]
eer_tpr = tpr[eer_idx]

fig, ax = plt.subplots(figsize=(6.5, 5.5))

ax.plot(fpr, tpr, color=C_TARGET, lw=2.5, label=f"GMM baseline (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random classifier (AUC = 0.5)")

# EER dot
ax.scatter([eer_far], [eer_tpr], color=C_PURPLE, s=80, zorder=5,
           label=f"EER operating point ({eer_oof*100:.1f}%)")
ax.annotate(
    f"  EER = {eer_oof*100:.1f}%",
    xy=(eer_far, eer_tpr),
    xytext=(eer_far + 0.06, eer_tpr - 0.08),
    fontsize=9, color=C_PURPLE,
    arrowprops=dict(arrowstyle="->", color=C_PURPLE, lw=1.2),
)

ax.set_xlabel("False Acceptance Rate (FAR)")
ax.set_ylabel("True Acceptance Rate (1 − FRR)")
ax.set_title("ROC Curve — OOF scores", fontweight="bold")
ax.legend()
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.show()

## 7. DET Curve (Detection Error Trade-off)

The **DET curve** plots FRR vs. FAR on a **probit (normal-deviate) scale**,
which linearises the curve for Gaussian score distributions and makes differences
near zero error rates visible.

- The diagonal dashed line is the **EER line** (FAR = FRR). Where the DET curve
  intersects this line is the EER operating point.
- A good system curves toward the bottom-left corner.
- The probit scale is standard in NIST speaker recognition evaluations.

In [ ]:
# DET uses probit-transformed FAR and FRR
far_det = fpr
frr_det = 1.0 - tpr

# Clip to avoid inf at 0 and 1
eps = 1e-6
far_p = stats.norm.ppf(np.clip(far_det, eps, 1 - eps))
frr_p = stats.norm.ppf(np.clip(frr_det, eps, 1 - eps))

# EER point on probit scale
eer_far_p = stats.norm.ppf(np.clip(eer_far, eps, 1 - eps))
eer_frr_p = stats.norm.ppf(np.clip(1.0 - eer_tpr, eps, 1 - eps))

# Axis tick positions: common operating rate values
tick_vals = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
tick_locs = stats.norm.ppf(tick_vals)
tick_labels = [f"{v*100:.1f}" for v in tick_vals]

fig, ax = plt.subplots(figsize=(6.5, 5.5))

ax.plot(far_p, frr_p, color=C_NONTARGET, lw=2.5, label="GMM baseline DET")

# EER diagonal
diag = np.linspace(far_p.min(), far_p.max(), 200)
ax.plot(diag, diag, "k--", lw=1, alpha=0.5, label="EER diagonal (FAR = FRR)")

# EER dot
ax.scatter([eer_far_p], [eer_frr_p], color=C_PURPLE, s=80, zorder=5,
           label=f"EER = {eer_oof*100:.1f}%")
ax.annotate(
    f"  EER = {eer_oof*100:.1f}%",
    xy=(eer_far_p, eer_frr_p),
    xytext=(eer_far_p + 0.4, eer_frr_p - 0.4),
    fontsize=9, color=C_PURPLE,
    arrowprops=dict(arrowstyle="->", color=C_PURPLE, lw=1.2),
)

ax.set_xticks(tick_locs)
ax.set_xticklabels(tick_labels, fontsize=8)
ax.set_yticks(tick_locs)
ax.set_yticklabels(tick_labels, fontsize=8)

ax.set_xlabel("FAR (%)")
ax.set_ylabel("FRR (%)")
ax.set_title("DET Curve — probit scale, OOF scores", fontweight="bold")
ax.legend()

# Clip view to the interesting region
lo = stats.norm.ppf(0.001)
hi = stats.norm.ppf(0.6)
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)

plt.tight_layout()
plt.show()

## 8. Per-fold EER Bar Chart

Variance across folds matters: a large standard deviation tells us the system is
sensitive to *which session* of the target speaker appears in the val set — possibly
because different sessions have different recording conditions.

Bars are coloured relative to the mean EER:
green = below mean (better), red = above mean (worse). The mean ± std is annotated.

In [ ]:
eers = results_df["eer"].to_numpy() * 100   # percent
fold_labels = [f"Fold {i}" for i in results_df["fold"].astype(int)]

# Colour: green if below mean, red if above
colors = [C_GREEN if e <= mean_eer * 100 else C_TARGET for e in eers]

fig, ax = plt.subplots(figsize=(7, 3.5))

bars = ax.barh(fold_labels, eers, color=colors, height=0.5, edgecolor="white", linewidth=0.8)

# Value labels inside / beside each bar
for bar, eer_val in zip(bars, eers):
    ax.text(
        bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
        f"{eer_val:.1f}%",
        va="center", ha="left", fontsize=10, fontweight="bold",
    )

# Mean line + annotation
ax.axvline(mean_eer * 100, color="#555", ls="--", lw=1.5, label=f"Mean EER")
ax.axvspan(
    (mean_eer - std_eer) * 100, (mean_eer + std_eer) * 100,
    alpha=0.12, color="#555",
)
ax.text(
    mean_eer * 100 + 0.3, len(eers) - 0.7,
    f"mean = {mean_eer*100:.1f}% ± {std_eer*100:.1f}%",
    fontsize=9, color="#333", va="top",
)

ax.set_xlabel("EER (%)")
ax.set_title("Per-fold EER — MFCC + GMM baseline", fontweight="bold")
ax.set_xlim(0, max(eers) * 1.25)
ax.invert_yaxis()   # fold 0 on top

plt.tight_layout()
plt.show()

## 9. Score Calibration Panel

The assignment asks for **hard decisions at prior 0.5**. The optimal threshold under
that prior (assuming perfectly calibrated log-likelihood ratios) is **0**.

In practice the LLR from GMMs is not perfectly calibrated, so the empirical
min-DCF threshold differs from zero. This panel shows:

- **Top subplot:** the score histogram with both threshold lines — ideal (black, threshold=0)
  and empirical min-DCF (green).
- **Bottom subplot:** cumulative error counts at each threshold — how many targets
  are missed (FN) and how many non-targets are falsely accepted (FA). The crossing
  point is the EER.

If the two thresholds are far apart the system needs score calibration (Platt scaling
or isotonic regression on OOF scores) before submission.

In [ ]:
s_t_all  = oof_scores[y_all == 1]
s_nt_all = oof_scores[y_all == 0]

ideal_thr   = 0.0
empirical_thr = thr_oof

# Count errors at ideal threshold
fn_ideal = int((s_t_all  < ideal_thr).sum())   # missed targets
fa_ideal = int((s_nt_all >= ideal_thr).sum())  # accepted non-targets

# Count errors at empirical threshold
fn_emp = int((s_t_all  < empirical_thr).sum())
fa_emp = int((s_nt_all >= empirical_thr).sum())

# Sweep of thresholds for error-rate curves
sweep = np.linspace(score_min, score_max, 300)
fn_sweep = np.array([(s_t_all  < t).sum() for t in sweep])
fa_sweep = np.array([(s_nt_all >= t).sum() for t in sweep])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# ── Top: histograms + thresholds ────────────────────────────────────────
bins2 = np.linspace(score_min, score_max, 40)
ax1.hist(s_nt_all, bins=bins2, alpha=0.6, color=C_NONTARGET,
         label=f"Non-target (n={len(s_nt_all)})", density=True)
ax1.hist(s_t_all,  bins=bins2, alpha=0.75, color=C_TARGET,
         label=f"Target (n={len(s_t_all)})", density=True)

ax1.axvline(ideal_thr, color="black", ls="-", lw=2,
            label=f"Ideal threshold = 0  (FN={fn_ideal}, FA={fa_ideal})")
ax1.axvline(empirical_thr, color=C_GREEN, ls="--", lw=2,
            label=f"min-DCF threshold = {empirical_thr:.2f}  (FN={fn_emp}, FA={fa_emp})")

ax1.set_ylabel("Density")
ax1.set_title("Score distributions with calibration thresholds", fontweight="bold")
ax1.legend(fontsize=8)

# Annotation: calibration gap
gap = empirical_thr - ideal_thr
ax1.annotate(
    "",
    xy=(empirical_thr, 0.01), xytext=(ideal_thr, 0.01),
    arrowprops=dict(arrowstyle="<->", color="#555", lw=1.5),
)
ax1.text(
    (ideal_thr + empirical_thr) / 2, 0.025,
    f"Δ = {gap:.2f}",
    ha="center", va="bottom", fontsize=9, color="#333",
)

# ── Bottom: cumulative error counts ─────────────────────────────────────
ax2.plot(sweep, fn_sweep, color=C_TARGET,    lw=2, label="Missed targets (FN)")
ax2.plot(sweep, fa_sweep, color=C_NONTARGET, lw=2, label="False acceptances (FA)")

ax2.axvline(ideal_thr,     color="black",  ls="-",  lw=1.8)
ax2.axvline(empirical_thr, color=C_GREEN,  ls="--", lw=1.8)

# Mark error counts at ideal threshold
ax2.scatter([ideal_thr], [fn_ideal], color=C_TARGET,    s=60, zorder=5)
ax2.scatter([ideal_thr], [fa_ideal], color=C_NONTARGET, s=60, zorder=5)
ax2.scatter([empirical_thr], [fn_emp], color=C_TARGET,    s=60, zorder=5, marker="D")
ax2.scatter([empirical_thr], [fa_emp], color=C_NONTARGET, s=60, zorder=5, marker="D")

ax2.set_xlabel("Decision threshold")
ax2.set_ylabel("Error count")
ax2.set_title("Cumulative errors vs. threshold", fontweight="bold")
ax2.legend(fontsize=9)
ax2.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.suptitle(
    f"Calibration gap: ideal thr = 0.00, empirical min-DCF thr = {empirical_thr:.2f}  "
    f"(Δ = {gap:.2f})",
    fontsize=11, fontweight="bold",
)
plt.tight_layout()
plt.show()

print(f"At ideal threshold (0.00): FN = {fn_ideal} / {len(s_t_all)} targets,  "
      f"FA = {fa_ideal} / {len(s_nt_all)} non-targets")
print(f"At min-DCF threshold ({empirical_thr:.2f}): FN = {fn_emp} / {len(s_t_all)} targets,  "
      f"FA = {fa_emp} / {len(s_nt_all)} non-targets")

## 10. Next Steps

Copy the numbers from the results table above into
`experiments/E001_audio-mfcc-gmm-baseline.md`:

- Per-fold EER/min-DCF table
- Interpretation: which fold was hardest and why (session effect?)
- Calibration gap — does the system need Platt scaling before submission?
- Next experiment: data augmentation (speed perturbation, noise addition) to
  reduce the per-fold variance and improve the worst-fold EER.